In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_15_5_2,0.999242,0.962341,0.999384,0.999617,0.999493,0.000752,0.037336,0.000799,0.000380,0.000589,0.059239,0.027417,1.000467,0.028584,140.386432,217.175609,"Hidden Size=[6, 4], regularizer=0.03, learning..."
1,model_15_5_3,0.999236,0.962311,0.998956,0.999466,0.999190,0.000758,0.037367,0.001354,0.000529,0.000942,0.055012,0.027527,1.000470,0.028699,140.370375,217.159552,"Hidden Size=[6, 4], regularizer=0.03, learning..."
2,model_22_8_15,0.999220,0.962755,0.999907,0.996847,0.998385,0.000773,0.036926,0.000088,0.003460,0.001774,0.017283,0.027811,1.000284,0.028995,194.329266,304.028091,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
3,model_22_8_14,0.999219,0.962636,0.999899,0.996978,0.998447,0.000774,0.037044,0.000096,0.003317,0.001706,0.017731,0.027822,1.000284,0.029007,194.327665,304.026490,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
4,model_22_8_16,0.999218,0.962866,0.999914,0.996716,0.998323,0.000775,0.036816,0.000081,0.003604,0.001843,0.016859,0.027845,1.000284,0.029031,194.324389,304.023213,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933,model_6_6_14,0.995849,0.961368,0.994850,0.993423,0.995004,0.004115,0.038301,0.002840,0.005326,0.004083,0.040251,0.064149,1.001811,0.066880,168.986165,265.277355,"Hidden Size=[6, 6], regularizer=0.5, learning_..."
945,model_6_9_0,0.995743,0.959727,0.999911,0.999952,0.999927,0.004220,0.039928,0.000086,0.000029,0.000057,0.044932,0.064962,1.001857,0.067727,168.935817,265.227007,"Hidden Size=[6, 6], regularizer=0.5, learning_..."
950,model_6_6_13,0.995694,0.962223,0.995371,0.994231,0.995581,0.004269,0.037453,0.002552,0.004671,0.003612,0.040169,0.065339,1.001879,0.068120,168.912693,265.203884,"Hidden Size=[6, 6], regularizer=0.5, learning_..."
957,model_14_6_5,0.995651,0.895411,0.997110,0.999450,0.999239,0.004311,0.103693,0.000782,0.000245,0.000514,0.084228,0.065661,1.002676,0.068457,136.892987,213.682164,"Hidden Size=[6, 4], regularizer=0.5, learning_..."
